In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 63 — Database Analyst Agent

## Goal
Build an agent that converts **natural language questions into SQL**,
runs them against a database, and **summarizes results** in plain English.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Text-to-SQL** | LLM generates SQL from English questions |
| **SQL tool** | Safe query execution with read-only guard |
| **Schema introspection** | Agent learns table structure first |
| **Result summarization** | Agent explains query results |

## Architecture
```
User: "Which department has the highest salary budget?"
  → Agent: get_schema() → write SQL → run_query() → summarize
```

## Stack
- `LangGraph` + `ChatOllama`
- `sqlite3` (in-memory database)
- Custom SQL tools with safety guards

In [ ]:
import os, warnings, sqlite3
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Create a Sample Database

We'll create an in-memory SQLite database with employee data
across departments, with salaries, hire dates, and performance ratings.

In [ ]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create tables
cursor.executescript("""
CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL,
    location TEXT
);

CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    dept_id INTEGER REFERENCES departments(dept_id),
    salary REAL,
    hire_date TEXT,
    performance_rating REAL
);

CREATE TABLE projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT,
    dept_id INTEGER REFERENCES departments(dept_id),
    budget REAL,
    status TEXT
);

INSERT INTO departments VALUES
    (1, 'Engineering', 'San Francisco'),
    (2, 'Marketing', 'New York'),
    (3, 'Sales', 'Chicago'),
    (4, 'HR', 'San Francisco'),
    (5, 'Finance', 'New York');

INSERT INTO employees VALUES
    (1, 'Alice Chen', 1, 145000, '2020-03-15', 4.5),
    (2, 'Bob Martinez', 1, 135000, '2019-07-01', 4.2),
    (3, 'Carol White', 1, 155000, '2018-01-10', 4.8),
    (4, 'David Kim', 2, 95000, '2021-06-20', 3.9),
    (5, 'Eve Johnson', 2, 88000, '2022-02-14', 4.1),
    (6, 'Frank Brown', 3, 110000, '2019-11-05', 4.6),
    (7, 'Grace Lee', 3, 105000, '2020-08-22', 3.7),
    (8, 'Henry Davis', 3, 98000, '2023-01-30', 4.0),
    (9, 'Ivy Wilson', 4, 85000, '2021-04-12', 4.3),
    (10, 'Jack Taylor', 4, 78000, '2022-09-01', 3.5),
    (11, 'Karen Moore', 5, 120000, '2018-05-20', 4.7),
    (12, 'Leo Anderson', 5, 115000, '2019-03-15', 4.4),
    (13, 'Mia Thomas', 1, 128000, '2021-10-01', 4.0),
    (14, 'Noah Jackson', 2, 92000, '2020-12-15', 3.8),
    (15, 'Olivia Harris', 3, 102000, '2022-06-10', 4.2);

INSERT INTO projects VALUES
    (1, 'Cloud Migration', 1, 500000, 'active'),
    (2, 'Brand Refresh', 2, 150000, 'completed'),
    (3, 'Q4 Campaign', 3, 200000, 'active'),
    (4, 'HR Portal', 4, 80000, 'active'),
    (5, 'Annual Audit', 5, 50000, 'completed'),
    (6, 'ML Platform', 1, 750000, 'active'),
    (7, 'Partner Program', 3, 120000, 'planned');
""")
conn.commit()

# Verify
for table in ["departments", "employees", "projects"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")
print("\n✅ Database created!")

## Step 2 — Define SQL Tools

Key safety: the `run_query` tool only allows **SELECT** statements.
No INSERT, UPDATE, DELETE, or DROP allowed.

In [ ]:
@tool
def get_schema() -> str:
    """Get the database schema — table names, columns, types, and sample rows."""
    tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    schema_info = []
    for (table_name,) in tables:
        cols = cursor.execute(f"PRAGMA table_info({table_name})").fetchall()
        col_desc = ", ".join(f"{c[1]} ({c[2]})" for c in cols)
        sample = cursor.execute(f"SELECT * FROM {table_name} LIMIT 3").fetchall()
        schema_info.append(f"Table: {table_name}\n  Columns: {col_desc}\n  Sample: {sample}")
    return "\n\n".join(schema_info)

@tool
def run_query(sql: str) -> str:
    """Execute a read-only SQL query against the database. Only SELECT statements are allowed. Returns results as formatted text."""
    # Safety: only allow SELECT
    sql_stripped = sql.strip().upper()
    if not sql_stripped.startswith("SELECT") and not sql_stripped.startswith("WITH"):
        return "ERROR: Only SELECT queries are allowed. No INSERT, UPDATE, DELETE, or DROP."
    
    dangerous_keywords = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "CREATE", "TRUNCATE"]
    for kw in dangerous_keywords:
        if kw in sql_stripped:
            return f"ERROR: Query contains forbidden keyword: {kw}"
    
    try:
        results = cursor.execute(sql).fetchall()
        if not results:
            return "Query returned no results."
        # Get column names
        col_names = [desc[0] for desc in cursor.description]
        header = " | ".join(col_names)
        rows = [" | ".join(str(v) for v in row) for row in results[:50]]
        return f"{header}\n{'─' * len(header)}\n" + "\n".join(rows)
    except Exception as e:
        return f"SQL Error: {e}"

@tool
def list_tables() -> str:
    """List all tables in the database with their row counts."""
    tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    info = []
    for (name,) in tables:
        count = cursor.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        info.append(f"  {name}: {count} rows")
    return "Tables:\n" + "\n".join(info)

sql_tools = [get_schema, run_query, list_tables]
print(f"Defined {len(sql_tools)} SQL tools")

## Step 3 — Create the SQL Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

SQL_SYSTEM_PROMPT = """You are a database analyst assistant.

Workflow:
1. First call get_schema() to understand the tables and columns
2. Write SQL queries to answer the user's question
3. Run queries with run_query()
4. Summarize results in plain English

Rules:
- Always check the schema before writing SQL
- Use proper JOINs when combining tables
- Explain the SQL you wrote and what it does
- Present results clearly with key numbers highlighted
- If a query fails, check the schema and try again"""

sql_agent = create_react_agent(
    model=llm,
    tools=sql_tools,
    prompt=SQL_SYSTEM_PROMPT,
)

def ask_sql_agent(question: str):
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")
    result = sql_agent.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:150]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:400]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 ANSWER:\n{msg.content}")
    return result

print("SQL agent ready!")

In [ ]:
# Question 1: Aggregation
ask_sql_agent("Which department has the highest total salary budget?")

In [ ]:
# Question 2: Join + filter
ask_sql_agent("List all employees working on active projects, with their department and project name.")

In [ ]:
# Question 3: Complex analysis
ask_sql_agent("Who are the top 3 highest-rated employees and how do their salaries compare to their department average?")

In [ ]:
# Question 4: Safety test — agent should refuse
ask_sql_agent("Delete all employees with a rating below 4.0")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Text-to-SQL** | LLM writes SQL from English questions |
| **Schema introspection** | Agent calls `get_schema()` to learn DB structure |
| **Read-only guard** | `run_query()` blocks INSERT/UPDATE/DELETE/DROP |
| **Multi-step reasoning** | Agent checks schema → writes SQL → explains results |

## Safety Pattern
```python
# Never let the LLM execute arbitrary SQL!
if not sql.startswith("SELECT"):
    return "ERROR: Only SELECT allowed"
```

## Production Considerations
- Use a **read-only database user** in production
- Add **query timeouts** to prevent expensive queries
- **Log all queries** for audit trail
- Consider **SQL injection** risks — parameterize when possible